In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

import torch


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ATML/PA1"
)

REPO_ROOT = Path(
    "/content/drive/MyDrive/ATML/PA1-repo"
)

TASK4_WORK_ROOT = (
    PROJECT_ROOT / "task4"
)

TASK4_REPO_ROOT = (
    REPO_ROOT / "task4"
)

DATA_ROOT = (
    PROJECT_ROOT / "datasets"
)

CACHE_ROOT = (
    TASK4_WORK_ROOT / "cache"
)

CHECKPOINT_PATH = (
    TASK4_WORK_ROOT
    / "checkpoints/vanilla_best.pt"
)

CIFAR10_SPLIT_PATH = (
    TASK4_REPO_ROOT
    / "splits/"
    "cifar10_train_val_seed6304.json"
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "Change the Colab runtime to GPU."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

print(
    "Vanilla checkpoint exists:",
    CHECKPOINT_PATH.exists(),
)

Device: cuda
GPU: Tesla T4
Vanilla checkpoint exists: True


In [3]:
%cd /content/drive/MyDrive/ATML/PA1-repo

import json

import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    DataLoader,
)

from common.seed import set_seed

from task4.data.cifar10 import (
    build_cifar10_output_datasets,
)

from task4.data.make_splits import (
    load_cifar10_split,
)

from task4.extract_outputs import (
    extract_model_outputs,
)

from task4.methods.vanilla import (
    build_vanilla_model,
)

from task4.scores.energy import (
    energy_unknownness,
)

from task4.scores.mahalanobis import (
    fit_shared_diagonal_gaussian,
    mahalanobis_unknownness,
)

from task4.scores.mls import (
    mls_unknownness,
)

from task4.scores.msp import (
    msp_unknownness,
)


SEED = 6304

set_seed(SEED)

CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Seed:", SEED)

/content/drive/.shortcut-targets-by-id/1tphHcVs5KfBbfa3Y2U6Gt0Lo7AWXysa4/ATML/PA1-repo
Seed: 6304


In [4]:
cifar10_split = (
    load_cifar10_split(
        CIFAR10_SPLIT_PATH
    )
)

output_datasets = (
    build_cifar10_output_datasets(
        data_root=DATA_ROOT,
        split=cifar10_split,
        download=True,
    )
)

output_loaders = {
    split_name: DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    for split_name, dataset
    in output_datasets.items()
    if split_name
    in {
        "train",
        "validation",
        "test",
    }
}

print(
    "Unaugmented training examples:",
    len(
        output_datasets[
            "train"
        ]
    ),
)

print(
    "Validation examples:",
    len(
        output_datasets[
            "validation"
        ]
    ),
)

print(
    "Test examples:",
    len(
        output_datasets[
            "test"
        ]
    ),
)

print(
    "CIFAR-100 loaded:",
    False,
)

Unaugmented training examples: 45000
Validation examples: 5000
Test examples: 10000
CIFAR-100 loaded: False


In [5]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model = build_vanilla_model().to(
    DEVICE
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

print(
    "Selected epoch:",
    checkpoint["epoch"],
)

print(
    "Selected validation accuracy:",
    checkpoint[
        "selection_score"
    ],
)

print(
    "All parameters frozen:",
    all(
        not parameter.requires_grad
        for parameter
        in model.parameters()
    ),
)

Selected epoch: 95
Selected validation accuracy: 0.9516
All parameters frozen: True


In [6]:
vanilla_outputs = {}

for split_name in [
    "train",
    "validation",
    "test",
]:
    output_path = (
        CACHE_ROOT
        / (
            f"vanilla_{split_name}_"
            "outputs.npz"
        )
    )

    extracted_outputs = (
        extract_model_outputs(
            model=model,
            data_loader=(
                output_loaders[
                    split_name
                ]
            ),
            device=DEVICE,
            description=(
                f"Vanilla {split_name}"
            ),
        )
    )

    np.savez_compressed(
        output_path,
        logits=(
            extracted_outputs[
                "logits"
            ]
        ),
        features=(
            extracted_outputs[
                "features"
            ]
        ),
        labels=(
            extracted_outputs[
                "labels"
            ]
        ),
        dataset_indices=(
            extracted_outputs[
                "dataset_indices"
            ]
        ),
    )

    vanilla_outputs[
        split_name
    ] = extracted_outputs

    print(
        split_name,
        "| logits:",
        extracted_outputs[
            "logits"
        ].shape,
        "| features:",
        extracted_outputs[
            "features"
        ].shape,
        "| saved:",
        output_path,
    )

Vanilla train:   0%|          | 0/176 [00:00<?, ?it/s]

train | logits: (45000, 10) | features: (45000, 512) | saved: /content/drive/MyDrive/ATML/PA1/task4/cache/vanilla_train_outputs.npz


Vanilla validation:   0%|          | 0/20 [00:00<?, ?it/s]

validation | logits: (5000, 10) | features: (5000, 512) | saved: /content/drive/MyDrive/ATML/PA1/task4/cache/vanilla_validation_outputs.npz


Vanilla test:   0%|          | 0/40 [00:00<?, ?it/s]

test | logits: (10000, 10) | features: (10000, 512) | saved: /content/drive/MyDrive/ATML/PA1/task4/cache/vanilla_test_outputs.npz


In [7]:
mahalanobis_estimator = (
    fit_shared_diagonal_gaussian(
        features=vanilla_outputs[
            "train"
        ]["features"],
        labels=vanilla_outputs[
            "train"
        ]["labels"],
        number_of_classes=10,
        epsilon=1e-6,
    )
)

estimator_path = (
    CACHE_ROOT
    / "vanilla_mahalanobis_estimator.npz"
)

np.savez_compressed(
    estimator_path,
    class_means=(
        mahalanobis_estimator[
            "class_means"
        ]
    ),
    diagonal_variance=(
        mahalanobis_estimator[
            "diagonal_variance"
        ]
    ),
    epsilon=np.array(
        [
            mahalanobis_estimator[
                "epsilon"
            ]
        ]
    ),
)

print(
    "Class means shape:",
    mahalanobis_estimator[
        "class_means"
    ].shape,
)

print(
    "Diagonal variance shape:",
    mahalanobis_estimator[
        "diagonal_variance"
    ].shape,
)

print(
    "Minimum variance:",
    mahalanobis_estimator[
        "diagonal_variance"
    ].min(),
)

print(
    "Estimator saved to:",
    estimator_path,
)

Class means shape: (10, 512)
Diagonal variance shape: (512,)
Minimum variance: 0.00025646304773078424
Estimator saved to: /content/drive/MyDrive/ATML/PA1/task4/cache/vanilla_mahalanobis_estimator.npz


In [8]:
def compute_vanilla_scores(
    outputs,
    mahalanobis_estimator,
):
    logits_tensor = torch.from_numpy(
        outputs["logits"]
    )

    return {
        "msp": (
            msp_unknownness(
                logits_tensor
            )
            .numpy()
        ),
        "mls": (
            mls_unknownness(
                logits_tensor
            )
            .numpy()
        ),
        "energy": (
            energy_unknownness(
                logits_tensor
            )
            .numpy()
        ),
        "mahalanobis": (
            mahalanobis_unknownness(
                features=outputs[
                    "features"
                ],
                estimator=(
                    mahalanobis_estimator
                ),
            )
        ),
    }


validation_scores = (
    compute_vanilla_scores(
        outputs=vanilla_outputs[
            "validation"
        ],
        mahalanobis_estimator=(
            mahalanobis_estimator
        ),
    )
)

score_thresholds = {
    score_name: float(
        np.percentile(
            scores,
            95,
        )
    )
    for score_name, scores
    in validation_scores.items()
}

validation_score_table = pd.DataFrame(
    {
        "dataset_index": (
            vanilla_outputs[
                "validation"
            ]["dataset_indices"]
        ),
        "label": (
            vanilla_outputs[
                "validation"
            ]["labels"]
        ),
        "predicted_label": (
            vanilla_outputs[
                "validation"
            ]["logits"].argmax(
                axis=1
            )
        ),
        **validation_scores,
    }
)

validation_score_path = (
    TASK4_REPO_ROOT
    / "results/"
    "vanilla_validation_scores.csv"
)

validation_score_table.to_csv(
    validation_score_path,
    index=False,
)

print("Validation thresholds:")

for score_name, threshold in (
    score_thresholds.items()
):
    acceptance_rate = np.mean(
        validation_scores[
            score_name
        ]
        <= threshold
    )

    print(
        score_name,
        "| threshold:",
        threshold,
        "| validation acceptance:",
        acceptance_rate,
    )

Validation thresholds:
msp | threshold: 0.16056357324123383 | validation acceptance: 0.95
mls | threshold: -5.903560161590576 | validation acceptance: 0.95
energy | threshold: -6.083648681640625 | validation acceptance: 0.95
mahalanobis | threshold: 3029.122777225638 | validation acceptance: 0.95


In [9]:
test_scores = (
    compute_vanilla_scores(
        outputs=vanilla_outputs[
            "test"
        ],
        mahalanobis_estimator=(
            mahalanobis_estimator
        ),
    )
)

known_score_rows = []

for score_name in [
    "msp",
    "mls",
    "energy",
    "mahalanobis",
]:
    threshold = score_thresholds[
        score_name
    ]

    validation_acceptance = float(
        np.mean(
            validation_scores[
                score_name
            ]
            <= threshold
        )
    )

    test_acceptance = float(
        np.mean(
            test_scores[
                score_name
            ]
            <= threshold
        )
    )

    known_score_rows.append(
        {
            "model": "vanilla",
            "score": score_name,
            "threshold": threshold,
            "validation_acceptance_rate": (
                validation_acceptance
            ),
            "known_test_acceptance_rate": (
                test_acceptance
            ),
            "known_test_rejection_rate": (
                1.0
                - test_acceptance
            ),
        }
    )

known_score_summary = pd.DataFrame(
    known_score_rows
)

known_summary_path = (
    TASK4_REPO_ROOT
    / "results/"
    "vanilla_known_score_summary.csv"
)

known_score_summary.to_csv(
    known_summary_path,
    index=False,
)

threshold_path = (
    TASK4_REPO_ROOT
    / "results/"
    "vanilla_score_thresholds.json"
)

with threshold_path.open(
    "w",
    encoding="utf-8",
) as threshold_file:
    json.dump(
        {
            "model": "vanilla",
            "percentile": 95,
            "calibration_split": (
                "cifar10_validation"
            ),
            "cifar100_used": False,
            "thresholds": (
                score_thresholds
            ),
        },
        threshold_file,
        indent=2,
    )

print(
    known_score_summary.to_string(
        index=False
    )
)

print()
print(
    "Thresholds saved to:",
    threshold_path,
)

  model       score   threshold  validation_acceptance_rate  known_test_acceptance_rate  known_test_rejection_rate
vanilla         msp    0.160564                        0.95                      0.9466                     0.0534
vanilla         mls   -5.903560                        0.95                      0.9474                     0.0526
vanilla      energy   -6.083649                        0.95                      0.9482                     0.0518
vanilla mahalanobis 3029.122777                        0.95                      0.9474                     0.0526

Thresholds saved to: /content/drive/MyDrive/ATML/PA1-repo/task4/results/vanilla_score_thresholds.json


In [10]:
posthoc_metadata = {
    "model": "vanilla",
    "checkpoint_epoch": int(
        checkpoint["epoch"]
    ),
    "scores": {
        "msp": (
            "1 - maximum softmax probability"
        ),
        "mls": (
            "negative maximum known-class logit"
        ),
        "energy": (
            "negative log-sum-exp of logits"
        ),
        "mahalanobis": (
            "minimum shared-diagonal "
            "Mahalanobis distance"
        ),
    },
    "mahalanobis": {
        "training_features": (
            "unaugmented CIFAR-10 "
            "training features"
        ),
        "number_of_classes": 10,
        "feature_dimension": 512,
        "covariance": (
            "shared diagonal"
        ),
        "diagonal_epsilon": 1e-6,
    },
    "threshold_calibration": {
        "split": (
            "CIFAR-10 validation"
        ),
        "percentile": 95,
        "accept_when": (
            "unknownness <= threshold"
        ),
    },
    "cifar100_used": False,
    "thresholds": (
        score_thresholds
    ),
    "known_test_acceptance": {
        row["score"]: float(
            row[
                "known_test_acceptance_rate"
            ]
        )
        for _, row
        in known_score_summary.iterrows()
    },
}

metadata_path = (
    TASK4_REPO_ROOT
    / "results/"
    "vanilla_posthoc_metadata.json"
)

with metadata_path.open(
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        posthoc_metadata,
        metadata_file,
        indent=2,
    )

print(
    json.dumps(
        posthoc_metadata,
        indent=2,
    )
)

{
  "model": "vanilla",
  "checkpoint_epoch": 95,
  "scores": {
    "msp": "1 - maximum softmax probability",
    "mls": "negative maximum known-class logit",
    "energy": "negative log-sum-exp of logits",
    "mahalanobis": "minimum shared-diagonal Mahalanobis distance"
  },
  "mahalanobis": {
    "training_features": "unaugmented CIFAR-10 training features",
    "number_of_classes": 10,
    "feature_dimension": 512,
    "covariance": "shared diagonal",
    "diagonal_epsilon": 1e-06
  },
  "threshold_calibration": {
    "split": "CIFAR-10 validation",
    "percentile": 95,
    "accept_when": "unknownness <= threshold"
  },
  "cifar100_used": false,
  "thresholds": {
    "msp": 0.16056357324123383,
    "mls": -5.903560161590576,
    "energy": -6.083648681640625,
    "mahalanobis": 3029.122777225638
  },
  "known_test_acceptance": {
    "msp": 0.9466,
    "mls": 0.9474,
    "energy": 0.9482,
    "mahalanobis": 0.9474
  }
}
